# 定增项目敏感性分析

## 分析目标
本 Notebook 用于分析定增项目对关键参数的敏感性，包括：
- 发行折价率
- 锁定期长度
- 解禁时股价表现
- 破发概率分析

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from utils.analysis_tools import PrivatePlacementRiskAnalyzer
from utils.font_config import setup_chinese_font

# 配置中文字体
setup_chinese_font()

%matplotlib inline
sns.set_style('whitegrid')

print('✅ 库导入成功')

## 1. 项目参数设置

In [ ]:
# 定增项目基本参数
PROJECT_PARAMS = {
    'issue_price': 20.0,      # 发行价格（元/股）
    'issue_shares': 5000000,  # 发行数量（股）
    'lockup_period': 12,      # 锁定期（月）
    'current_price': 18.5,    # 当前价格（元/股）
    'risk_free_rate': 0.03    # 无风险利率
}

# 创建分析器实例
analyzer = PrivatePlacementRiskAnalyzer(**PROJECT_PARAMS)

# 显示项目摘要
summary = analyzer.generate_summary_report()
print('=== 项目摘要 ===')
print(f"发行价格: {summary['project_info']['issue_price']} 元/股")
print(f"当前价格: {summary['project_info']['current_price']} 元/股")
print(f"折价率: {summary['key_metrics']['discount_to_current']}")
print(f"融资金额: {summary['project_info']['issue_amount']/10000:.2f} 万元")
print(f"锁定期: {summary['project_info']['lockup_period_months']} 个月")

## 2. 盈亏平衡价格敏感性

In [ ]:
# 不同期望收益率下的盈亏平衡价格
target_returns = np.linspace(0.05, 0.50, 10)  # 5%到50%的年化收益率
break_even_prices = [analyzer.calculate_break_even_price(r) for r in target_returns]

# 绘制敏感性曲线
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(target_returns * 100, break_even_prices, 'b-', linewidth=2, label='盈亏平衡价格')
ax.axhline(y=PROJECT_PARAMS['issue_price'], color='r', linestyle='--', label='发行价格')
ax.axhline(y=PROJECT_PARAMS['current_price'], color='g', linestyle='--', label='当前价格')

ax.set_xlabel('期望年化收益率 (%)', fontsize=12)
ax.set_ylabel('盈亏平衡价格 (元)', fontsize=12)
ax.set_title('不同收益率要求下的盈亏平衡价格', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 打印关键数据点
print('\n=== 盈亏平衡价格分析 ===')
for ret in [0.10, 0.15, 0.20, 0.25, 0.30]:
    be_price = analyzer.calculate_break_even_price(ret)
    distance = (analyzer.current_price - be_price) / analyzer.current_price
    print(f"{ret*100:.0f}%年化收益率: 盈亏平衡={be_price:.2f}元, 距离当前价={distance*100:.2f}%")

## 3. 锁定期敏感性分析

In [ ]:
# 不同锁定期下的盈亏平衡价格
lockup_periods = [6, 12, 18, 24, 36]  # 不同的锁定期（月）
target_return = 0.20  # 假设目标年化收益率20%

results = []
for period in lockup_periods:
    temp_analyzer = PrivatePlacementRiskAnalyzer(
        issue_price=PROJECT_PARAMS['issue_price'],
        issue_shares=PROJECT_PARAMS['issue_shares'],
        lockup_period=period,
        current_price=PROJECT_PARAMS['current_price']
    )
    be_price = temp_analyzer.calculate_break_even_price(target_return)
    results.append({
        'lockup_period': period,
        'break_even_price': be_price,
        'required_increase': (be_price - analyzer.current_price) / analyzer.current_price
    })

df_lockup = pd.DataFrame(results)

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(df_lockup['lockup_period'], df_lockup['break_even_price'], color='steelblue', alpha=0.7)
ax1.axhline(y=PROJECT_PARAMS['current_price'], color='r', linestyle='--', label='当前价格')
ax1.set_xlabel('锁定期 (月)')
ax1.set_ylabel('盈亏平衡价格 (元)')
ax1.set_title('锁定期对盈亏平衡价格的影响')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(df_lockup['lockup_period'], df_lockup['required_increase']*100, 'ro-', linewidth=2, markersize=8)
ax2.set_xlabel('锁定期 (月)')
ax2.set_ylabel('需要涨幅 (%)')
ax2.set_title('锁定期与需要涨幅的关系')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('=== 锁定期敏感性分析 ===')
print(df_lockup.to_string(index=False))

## 4. 破发概率分析

In [ ]:
# 假设股价服从对数正态分布，计算不同锁定期后的破发概率
volatilities = [0.2, 0.3, 0.4, 0.5]  # 不同的年化波动率

breakdown_results = []
for vol in volatilities:
    # 计算盈亏平衡价格
    be_price = analyzer.calculate_break_even_price(0.20)
    
    # 计算达到盈亏平衡价的概率
    prob = analyzer.calculate_probability_above_price(
        current_price=analyzer.current_price,
        volatility=vol,
        target_price=be_price,
        period_months=analyzer.lockup_period
    )
    
    breakdown_results.append({
        'volatility': vol,
        'break_even_price': be_price,
        'profit_probability': prob,
        'loss_probability': 1 - prob
    })

df_prob = pd.DataFrame(breakdown_results)

# 可视化
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df_prob['volatility']*100, df_prob['profit_probability']*100, 
        'g-o', linewidth=2, markersize=10, label='盈利概率')
ax.plot(df_prob['volatility']*100, df_prob['loss_probability']*100, 
        'r-s', linewidth=2, markersize=10, label='破发概率')
ax.set_xlabel('年化波动率 (%)')
ax.set_ylabel('概率 (%)')
ax.set_title('不同波动率下的盈亏概率分析', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n=== 破发概率分析 ===')
print(df_prob.to_string(index=False))

## 5. 龙卷风图 - 参数敏感性排序

In [ ]:
# 关键参数及其变化范围
tornado_params = {
    '发行折价率': (-0.3, 0.3),
    '锁定期(月)': (6, 36),
    '年化波动率': (0.2, 0.6),
    '无风险利率': (0.02, 0.05)
}

# 计算每个参数的影响（简化示例）
param_impacts = []
base_value = analyzer.calculate_break_even_price(0.20)

for param, (min_val, max_val) in tornado_params.items():
    # 这里简化处理，实际应该重新计算模型
    impact = abs(max_val - min_val) / base_value * 100
    param_impacts.append({
        'parameter': param,
        'impact_percent': impact
    })

df_tornado = pd.DataFrame(param_impacts).sort_values('impact_percent', ascending=True)

# 绘制龙卷风图
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(df_tornado['parameter'], df_tornado['impact_percent'], color='coral')
ax.set_xlabel('对收益率的影响 (%)')
ax.set_title('参数敏感性排序（龙卷风图）', fontsize=14, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)

# 添加数值标签
for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.5, bar.get_y() + bar.get_height()/2,
            f'{width:.1f}%', ha='left', va='center')

plt.tight_layout()
plt.show()

print('\n=== 参数敏感性排序 ===')
print(df_tornado.to_string(index=False))

## 6. 敏感性分析结论

In [ ]:
print('\n' + '='*50)
print('敏感性分析结论')
print('='*50)
print(f"\n1. 当前价格较发行价: {(analyzer.current_price/analyzer.issue_price - 1)*100:.2f}%")
print(f"\n2. 20%年化收益率要求下的盈亏平衡价: {analyzer.calculate_break_even_price(0.20):.2f}元")
print(f"\n3. 当前价距离盈亏平衡价: {(analyzer.current_price - analyzer.calculate_break_even_price(0.20))/analyzer.current_price*100:.2f}%")
print(f"\n4. 在30%年化波动率假设下，盈利概率约: {df_prob[df_prob['volatility']==0.3]['profit_probability'].values[0]*100:.1f}%")
print(f"\n5. 最敏感的参数是: {df_tornado.iloc[-1]['parameter']}")